# API Domonstration


### Import libs


In [ ]:
from dataclasses import asdict

from pyproj import Transformer
import cv2
import numpy as np
import matplotlib.pyplot as plt

from uavcalibration.datasets import *
from uavcalibration.map import *
from uavcalibration.calibrate import Calibrator

### Read uav data


In [ ]:
dataset = VisLocDataset(r"..\datasets\UAV_VisLoc_example")
satellite_map: Map
satellite_map = GeoTiffMap([i.image_path for i in dataset.satellite_infos.values()])
# satellite_map = TileMap(r"http://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}")
uav_data = dataset[0]
plt.imshow(uav_data.uav_image)

### Coarse calibration (perspective transform)


In [ ]:
calibration = Calibrator(uav_data.uav_image)
calibration.coarse_calibrate(**asdict(uav_data))
src_shape = uav_data.uav_image.shape
calibration.transform.adjust_shape(src_shape=(src_shape[1], src_shape[0]))

### Fetch satellite image and transform


In [ ]:
tmp_transform = calibration.transform.combined
h, w = calibration.uav_image.shape[:2]
async with satellite_map:
    satellite_image, satellite_crs = await satellite_map.get_async(
        tmp_transform.bounds(h=h, w=w),
        tmp_transform.crs,
        resolution=tmp_transform.resolution,
    )
plt.imshow(satellite_image)
satellite_crs

### Fine calibration process (image matching)


In [ ]:
calibration.fine_calibrate(
    satellite_image=satellite_image, satellite_crs=satellite_crs, tolerance=5, plot=True
)

### Get target transform


In [ ]:
print(calibration.transform.combined)